# STATS 425 Fall 2026 · Week 1 discussion
**Next-token probabilities, loss, and perplexity**  
**TA:** Ali Kaazempur-Mofrad

Today, we measure how well a pretrained language model predicts text. We obtain its next-token probabilities, convert them into token losses, and summarize those losses with perplexity.

| Section | Question we answer |
|---|---|
| 1. Loss and perplexity: a worked example | How do we calculate loss and perplexity from given probabilities? |
| 2. Loading a pretrained model and tokenizing text | How do we load a pretrained model and convert text to token IDs? |
| 3. Next-token probabilities | What probabilities does the model assign to possible next tokens? |
| 4. Calculating sentence loss and perplexity | How do we combine token probabilities into sentence loss and perplexity? |


**Set up the notebook.** The cell below installs the library versions used in these labs, and installs PyTorch only if it is missing.

In [ ]:
# Setup
import importlib.util, subprocess, sys
packages = ["transformers==4.57.6", "huggingface-hub==0.36.2", "tokenizers==0.22.2"]
if importlib.util.find_spec("torch") is None:
    packages.append("torch==2.8.0")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Setup complete.")

Setup complete.


## 1 · Loss and perplexity: a worked example
**Our first question:** how do we calculate loss and perplexity from probabilities?

Suppose a model assigns probabilities **0.5, 0.25, and 0.125** to the actual next tokens at three different positions. Each probability comes from a different context, so these three values do not need to sum to one.

Throughout this notebook, we follow three steps:

1. **Per token:** calculate its loss, $-\ln(p)$.
2. **Across tokens:** average those losses.
3. **Perplexity:** take the exponential of the average loss, $e^{\text{mean loss}}$.

$$\text{mean loss}=\frac{\text{sum of token losses}}{\text{number of scored tokens}},\qquad\mathrm{PPL}=\exp(\text{mean loss}).$$

A lower probability for the observed token gives a larger loss. Using natural logarithms gives a mean loss measured in **nats per token**. With base-2 logs, use $2^{\text{mean loss}}$ instead.

In [30]:
import math

# Consider probabilities for the observed tokens at three different positions.
probabilities = [0.5, 0.25, 0.125]
token_losses = [-math.log(p) for p in probabilities]

print(f"{'Target':>6s} {'Probability':>13s} {'Token loss: -ln(p)':>20s}")
for position, (p, loss) in enumerate(zip(probabilities, token_losses), start=1):
    print(f"{position:6d} {p:13.6f} {loss:20.6f}")

total_nll = sum(token_losses)
mean_nll = total_nll / len(token_losses)
ppl = math.exp(mean_nll)
print(f"\nTotal loss = {' + '.join(f'{loss:.6f}' for loss in token_losses)}")
print(f"           = {total_nll:.6f}")
print(f"Mean loss  = {total_nll:.6f} / {len(token_losses)} = {mean_nll:.6f} nats/token")
print(f"Perplexity = exp({mean_nll:.6f}) = {ppl:.4f}")

Target   Probability   Token loss: -ln(p)
     1      0.500000             0.693147
     2      0.250000             1.386294
     3      0.125000             2.079442

Total loss = 0.693147 + 1.386294 + 2.079442
           = 4.158883
Mean loss  = 4.158883 / 3 = 1.386294 nats/token
Perplexity = exp(1.386294) = 4.0000


A perplexity of 4 corresponds to the same mean loss as assigning probability 1/4 to every observed token. It does not mean “four errors” or “25% accuracy.”

**Try it:** use `[0.25, 0.25, 0.25]`, then make one probability very small. Predict how the result will change, and compare the printed values with your own calculation.

<br>

Sections 2–3 show how to obtain probabilities from a pretrained model. In Section 4, we will use those probabilities to repeat the calculation from Section 1.

## 2 · Loading a pretrained model and tokenizing text
**Building on Section 1:** we know how to calculate loss from probabilities. To obtain probabilities from a model, we first need to convert our text into token IDs.

<br>

We will use the same sentence in Sections 2–4: **“The capital of France is Paris.”**

1. **Load the tokenizer and pretrained model.** The tokenizer converts text into token IDs. The model contains weights learned during earlier training.
2. **Display the sentence's tokens.** Each row shows a token's position, its ID, and the text it represents. We are preparing the input; we have not made any predictions yet.


<details><summary>Model-loading details and options</summary>

`from_pretrained(...)` loads the saved tokenizer or model. `REVISION` selects a specific version of those files. `.to(device)` moves the model to the CPU or GPU. Setting a seed helps make random operations repeatable. `model.eval()` turns off training behaviors such as dropout; it does not update or permanently freeze the weights. EOS means “end of sequence.” Padding fills unused positions when sequences of different lengths are grouped into a batch.

</details>

In [ ]:
import os, random, time
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 425
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))

MODEL_ID = "distilbert/distilgpt2"
REVISION = "2290a62682d06624634c1f46a6ad5be0f47f38aa"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, use_safetensors=True
).to(device)
# GPT-2 has no dedicated PAD token. Mask padding by position, not by token ID.
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model.config.pad_token_id = tokenizer.pad_token_id
model.eval()
print("Device:", device, "| torch:", torch.__version__, "| transformers:", transformers.__version__)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Loaded pretrained weights; no optimizer step has run.")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Device: cpu | torch: 2.11.0+cpu | transformers: 5.16.1
Parameters: 81,912,576
Loaded pretrained weights; no optimizer step has run.


The output above shows the device (CPU or GPU), library versions, and number of model parameters. Loading pretrained weights does not train the model.

**Convert text to token IDs.** `encode` converts the sentence into token IDs. `decode` converts those IDs back into text so we can see what each token represents.

In [ ]:
text = "The capital of France is Paris."
ids = tokenizer.encode(text, add_special_tokens=False)

print("Sentence:", text)
print(f"\n{'Position':>8s} {'Token ID':>9s}  Token text")
for position, token_id in enumerate(ids):
    print(f"{position:8d} {token_id:9d}  {tokenizer.decode([token_id])!r}")
print("\nDecoded again:", tokenizer.decode(ids))

Sentence: The capital of France is Paris.

Position  Token ID  Token text
       0       464  'The'
       1      3139  ' capital'
       2       286  ' of'
       3      4881  ' France'
       4       318  ' is'
       5      6342  ' Paris'
       6        13  '.'

Decoded again: The capital of France is Paris.


**Interpreting the output:** the **position** tells us where a token occurs in the sentence. Its **token ID** is an index in the tokenizer's vocabulary. Neither number is a probability, and a larger ID does not mean a more important token.

<br>

**Why do some tokens include a space?** In `' capital'`, the space before `capital` is part of the token. It keeps the words separated: joining `'The'` and `' capital'` gives `'The capital'`. Without the space, we would get `'Thecapital'`.

The quotes are added only when we print the tokens, to make spaces easier to see; the quotes themselves are not part of the tokens.

For the tokenizer used in this notebook, the space changes the token ID:

| Token text | Token ID |
|---|---:|
| `'capital'` | 27544 |
| `' capital'` | 3139 |

These are **two different entries in the tokenizer's vocabulary**. The second token represents the space and the word together, so the space does not need a separate token here. These IDs are specific to this tokenizer; other tokenizers may split the same text differently.

<br>

Our sentence has seven tokens, including `' Paris'` and `'.'`. Other words can split into several pieces, so a token is not always a whole word. The final line converts all the IDs back into the original sentence, including its spaces.

<br>

Next, we will give the model the first five token IDs, representing `'The capital of France is'`, and inspect the probability it assigns to the actual next token, `' Paris'`. We identify that token from our sentence.

## 3 · Next-token probabilities
We now give the model the token IDs for the beginning of our sentence. This preceding text is the **context** the model uses to predict the next token.

```text
Full sentence:       The capital of France is Paris.
Context shown:       The capital of France is
Actual next token:  ' Paris'
```

For the next position, the model returns one score (a **logit**) for every token in its vocabulary. **Softmax** converts these scores into probabilities. We display the five most likely tokens, then look up the probability assigned to the **actual next token**, `' Paris'`, even if it is not among those five.

The last line computes `-ln(probability)` for this observed token, using the formula from Section 1. This gives us one token loss. In Section 4, we will calculate the losses for the other tokens and combine them.

<details><summary>How the code selects a next-token probability</summary>

In this sentence, `len(ids) - 2` locates `' Paris'`, the second-to-last token. `ids[:target_position]` contains only the preceding context. A tensor is an array of numbers; `[prefix_ids]` gives a batch containing one sequence.

The logits have shape `[batch, positions, vocabulary]`. `[0, -1]` selects the first sequence and final context position. `.softmax(dim=-1)` converts the scores into probabilities that sum to one over the vocabulary. `.topk(5)` finds the five largest entries. `probs[observed_id]` selects the entry we need for scoring the actual text.

</details>

In [ ]:
# Use Section 2's IDs. In this sentence, the second-to-last token is " Paris".
target_position = len(ids) - 2
prefix_ids = ids[:target_position]
observed_id = ids[target_position]
prefix = tokenizer.decode(prefix_ids)
input_ids = torch.tensor([prefix_ids], device=device)

model.eval()
with torch.no_grad():
    probs = model(input_ids=input_ids).logits[0, -1].softmax(dim=-1)

print("Context:", repr(prefix))
print("Actual next token in our sentence:", repr(tokenizer.decode([observed_id])))
print("\nFive most likely next tokens:")
values, indices = probs.topk(5)
for token_id, probability in zip(indices.tolist(), values.tolist()):
    print(f"{tokenizer.decode([token_id])!r:20s} {probability:.8g}")

section3_probability = probs[observed_id].item()
print(f"\nProbability of the ACTUAL next token: {section3_probability:.8g}")
print(f"Its token loss: -ln({section3_probability:.8g}) = {-math.log(section3_probability):.6f}")

Context: 'The capital of France is'
Actual next token in our sentence: ' Paris'

Five most likely next tokens:
' the'               0.1177453
' home'              0.082270361
' now'               0.049867146
' a'                 0.049333956
' being'             0.035612348

Probability of the ACTUAL next token: 0.0027634266
Its token loss: -ln(0.0027634266) = 5.891284
Section 4 repeats this calculation for every scored token in the sentence.


The top-five list shows the tokens the model considers most likely to come next. The final probability and loss refer to the token that **actually comes next in our sentence**. We use that token's probability when calculating perplexity, even if the model prefers another token.

If `' Paris'` is absent from the top five, it still has a probability. This small pretrained model may prefer continuations like `' home'` or `' the'`. It was trained to predict text continuations, so we should not expect it to reliably answer factual questions.

## 4 · Calculating sentence loss and perplexity
Section 2 gave us the token IDs. Section 3 showed the probability of one actual next token. Now we find the probability of each token we can score, then calculate the mean loss and perplexity as we did in Section 1.

The table includes the `' Paris'` prediction from Section 3. It also scores `' capital'`, `' of'`, `' France'`, `' is'`, and the final period.

<br>

**Why are there six rows for seven tokens?** We do not add a special token before the sentence, so there is no preceding position from which to predict the first token (`'The'`). We therefore do not score that first token in this example. Each remaining token has a preceding context. The model internally aligns each position's prediction with the next token when we supply `labels=sentence_ids`.

<br>

After the table, the cell prints the complete calculation:

$$\text{total loss}=\sum_t -\ln(p_t),\quad\text{mean loss}=\frac{\text{total loss}}{6},\quad\mathrm{PPL}=\exp(\text{mean loss}).$$

The **model's built-in loss** should match the mean loss we calculated. Compare the two printed values.

<details><summary>How one model pass scores every position</summary>

A causal model can calculate all these next-token distributions together while preventing each position from seeing future tokens. `log_softmax` computes log probabilities directly, which helps avoid numerical problems with very small probabilities. `log_probs[position - 1, ids[position]]` selects the log probability assigned to the actual next token.

Using the actual preceding tokens as context is called **teacher forcing**. The target tokens come from the text itself, so this loss can be used for **self-supervised learning**. Here, we only calculate the loss; we do not update the model weights.

</details>

In [ ]:
# Reuse the full sentence and IDs from Section 2.
sentence_ids = torch.tensor([ids], device=device)
model.eval()
with torch.no_grad():
    result = model(input_ids=sentence_ids, labels=sentence_ids)
    # Position t predicts token t+1. log_softmax computes log probabilities stably.
    log_probs = result.logits[0, :-1].log_softmax(dim=-1)

rows = []
for position in range(1, len(ids)):
    context = tokenizer.decode(ids[:position])
    actual_token = tokenizer.decode([ids[position]])
    log_p = log_probs[position - 1, ids[position]].item()
    probability = math.exp(log_p)
    token_loss = -log_p
    rows.append((context, actual_token, probability, token_loss))

print("Sentence:", repr(text))
print(f"\n{'Context available to the model':<34s} {'Actual token':<14s} {'Probability':>13s} {'-ln(p)':>12s}")
for context, actual_token, probability, token_loss in rows:
    print(f"{context!r:<34s} {actual_token!r:<14s} {probability:13.8g} {token_loss:12.6f}")

token_losses = [row[3] for row in rows]
total_nll = sum(token_losses)
num_targets = len(token_losses)
mean_nll = total_nll / num_targets
ppl = math.exp(mean_nll)

print(f"\nTotal loss = {' + '.join(f'{loss:.6f}' for loss in token_losses)}")
print(f"           = {total_nll:.6f}")
print(f"Mean loss  = {total_nll:.6f} / {num_targets} = {mean_nll:.6f} nats/token")
print(f"Perplexity = exp({mean_nll:.6f}) = {ppl:.4f}")
print(f"\nModel's built-in loss: {result.loss.item():.6f} (compare with our mean loss)")
print(f"Scored targets: {num_targets}; input tokens: {len(ids)}")
print(f"\nParis row probability here: {rows[target_position - 1][2]:.6f}")
print(f"Paris probability in Section 3: {section3_probability:.6f}")
print("Calculations use full precision; displayed values are rounded.")

Sentence: 'The capital of France is Paris.'

Context available to the model     Actual token     Probability       -ln(p)
'The'                              ' capital'     6.1598272e-07    14.300047
'The capital'                      ' of'             0.38619924     0.951402
'The capital of'                   ' France'       0.0098528254     4.619997
'The capital of France'            ' is'            0.049358759     3.008640
'The capital of France is'         ' Paris'        0.0027634262     5.891284
'The capital of France is Paris'   '.'               0.27437827     1.293248

Total loss = 14.300047 + 0.951402 + 4.619997 + 3.008640 + 5.891284 + 1.293248
           = 30.064617
Mean loss  = 30.064617 / 6 = 5.010770 nats/token
Perplexity = exp(5.010770) = 150.0201

Model's built-in loss: 5.010769 (compare with our mean loss)
Scored targets: 6; input tokens: 7

Paris row probability here: 0.002763
Paris probability in Section 3: 0.002763
Calculations use full precision; displayed values a

Read each row of the output table as:

> “After this context, the actual next token was ____. The model assigned it probability ____, so its loss is $-\ln(p)$ = ____.”

Compare the `' Paris'` row with Section 3, and compare the mean loss with the model's built-in loss. Small display differences can come from rounding; the calculations use full precision.

## Recap

We followed one sentence through the complete calculation:

**Text → token IDs → next-token probabilities → token losses → mean loss → perplexity**

The token IDs identify the text. The model supplies probabilities. We select the probability of each **observed** next token, compute $-\ln(p)$, average those losses, and take the exponential of the average. Sections 1 and 4 used the same arithmetic; Section 4 obtained its probabilities from a real model.


### Check your understanding

1. Is the token ID for `' Paris'` its probability? What does the ID represent?
2. If the model's favorite next token is `' the'` but the text says `' Paris'`, which probability contributes to the loss?
3. If the model assigns probability 0.2 to every scored token, what is the perplexity?
4. Why did our seven-token sentence have six scored targets?
5. If one observed token becomes less likely while the other probabilities stay fixed, what happens to the mean loss and PPL?

<details><summary>Answers · open after discussing</summary>

1. No. It is an index in the tokenizer's vocabulary. The model supplies probabilities separately.
2. The probability of `' Paris'`, the actual observed token, even if it is outside the top five.
3. Mean loss is $-\ln(0.2)=\ln(5)$, so PPL is 5.
4. We did not add a special token before the sentence, so we did not score the first token. Each of the remaining six tokens has a preceding context.
5. That token's loss increases, so the mean loss and PPL both increase.

</details>


## References and further reading
- Cesa-Bianchi & Lugosi, *Prediction, Learning, and Games*, §9.1, printed pp. 247–248 (PDF pages 261–262): sequential probability assignment and logarithmic loss. This reading explains the mathematics behind the loss calculation.
- [Hugging Face: perplexity](https://huggingface.co/docs/transformers/perplexity): tokenization, context windows, and measuring perplexity.
- [GPT-2 API documentation](https://huggingface.co/docs/transformers/v4.57.3/en/model_doc/gpt2): causal model outputs and internally shifted labels.
- [DistilGPT2 model card](https://huggingface.co/distilbert/distilgpt2): checkpoint provenance and intended uses.